Welcome!
In this page, we'll have a walk-through about the steps of RandomForst model with Titanic Competition!

In this tutorial, we will learn...
    * Random Forest Classifier
    * Preprocessing and Pipeline
    * GridSearchCV
    * The concepts of splitting data
    
They are the essentials of machine learning which can be applied in many other learning algorithms (i.g. Linear Regression, Decision Tree, etc)

# Part 1 : Reading The Question Carefully

In a nutshell, this Titanic Competition is asking : 
"A passenger in the Titanic was:
    - male,
    - In the first class,
    - 32 years old
    - (and so on)
    would he survive in the Titanic?"

This is a typical classification problem where we put 0 for non-survivors and 1 for the survivors. 
With that, we can change the question into "Return a csv that is filled with ones and zeros to tell if the passenger has survived."

Let's have a peek of the data to see about it.   

In [33]:
import pandas as pd

data = pd.read_csv('/kaggle/input/titanic/train.csv')
data.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [34]:
percentages = data.groupby('Sex')['Survived'].mean() * 100
percentages

print(f"Female survived: {percentages['female']:.2f}%")
print(f"Male survived: {percentages['male']:.2f}%")

Female survived: 74.20%
Male survived: 18.89%


On the first code block, we can see features of the data and type of each column.

In here, we have 
    * Survived : Survival of the passenger
    * Pclass : Class of the passenger (1 = 1st, 2 = 2nd, 3 = 3rd)
    * SibSp : The number of siblings / spouses aboard the Titanic	
    * (and so on)

In there, you can see the full description of the data. https://www.kaggle.com/competitions/titanic/data

As you can see, some feature is integer and some are string, and some are correlated and some are not. 
In the next part, we'll preprocess the data to sort these chaos. 

# Part 2 : Pruning the branches

Most of the case, it's "Mo' data, Mo' accuracy", but if you take a look at the table, there's useless feature on the data.
The question is to predict the survival of the passenger but Name, Ticket, Cabin, Passenger won't do much of the work because how they aren't really corrlated to survival. So the first pruning we'll have on this data is removing those columns.

In the code below, we're dropping those features and saving it.
(We're saving 'Survived' in somewhere else since we don't want the model to train on the answer)

In [35]:
y = data['Survived']
data.drop(['Survived', 'Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)

In machine learning, we never put  target data (i.e. Survived) in the training set. That is to let the machine to learn the feature (i.e. Sex, Age), and apparently, having the 'answer' in the training data would make the model to memorize the answer instead of learning the actual features.

We'll use sklearn's train_test_split for the following benefits.
    * Randomization: it shuffles data before splitting — avoids patterns (like all survivors being at the top of the file).
    * Balanced split: keeps data distribution more even between train/test.
    * Convenience: handles both features (X) and labels (y) together, so they stay matched.

The biggest reason we're not just slicing through is for the randomization. You can think this train_test_splt as a mixing the fried rice, where each scoop get you adequate amout of rice, bean, and chicken. And slicing as a just solid fried rice. It might get you good balanced scoop just like the mixed one, but it's less likely to be.  

In [36]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(data, y, test_size=0.2, random_state=39)

train_test_split expects those variables in order. X_train, X_valid, y_train, y_valid.
Those 'valid' will be explained later on grid search.

In [37]:
data['Embarked'].unique()

array(['S', 'C', 'Q', nan], dtype=object)

In the next step, we have encoding here. Encoding is a process that changes the data type into another to be used more effectively. The encoding algorithm we'll use in this model is One-Hot Encoding.

Since models does not take strings as the input (except SVM), we turn string into numbers. And it can be done in this way: S -> 0, C -> 1, Q -> 2. This is called Label Encoding. But what can happen now is that machine can think S < C < Q.
One-Hot Encoding is an algorithm that prevents this confusion where the model takes ordinal value numerically.

So what OneHotEncoder does is that it makes N new rows where the N is the number of unique value in the feature and for each rows.
In each row, it's going to be a checkbox - if the value was S, the value of Embarked_S is going to be 1 in that row. 

Using OneHotEncoder manually is ugly so we'll use pipeline to make it convenient. 

In [40]:
from sklearn.compose import ColumnTransformer

# Selects all categorical columns
categorical_columns = X_train.select_dtypes(include=["category", "object"]).columns


preprocessor = ColumnTransformer([
  ('categorical', OneHotEncoder(handle_unknown="ignore"), categorical_columns)
])

The categorical_columns is where we can select all string features. Printing it will give us ["Sex", "Embarked"].
The preprocessor will be joined as a whole model as the classifier is defind in further steps.
It'll be looking like this :

model = Pipeline([
  ('process', preprocessor),
  ('classifier', classifier)
])

The code below shows manual implementation of OneHotEncoder. You can check that out to see the precedure in code. 
**That is not for out project right now, so don't write it down.**


In [38]:
from sklearn.preprocessing import OneHotEncoder


# Encoding Process
oneHotEncoder = OneHotEncoder(sparse_output=False)
data_oneHot = oneHotEncoder.fit_transform(data[["Embarked"]]) # We are transforming Embarked here
data_oneHot  # This returns dense array (Search 'Sparse Array' for more information)

# Turning it to DataFrame
cols = oneHotEncoder.get_feature_names_out(["Embarked"])  # Array that contains the names of feature.
data_oneHot_df = pd.DataFrame(data_oneHot, columns=cols, index=data.index)

# Joining it to the original data
data = pd.concat([data.drop("Embarked", axis=1), data_oneHot_df], axis=1)
data[['Embarked_C', 'Embarked_Q', 'Embarked_S', 'Embarked_nan']].head(6)

,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan
0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0
3,0.0,0.0,1.0,0.0
4,0.0,0.0,1.0,0.0
5,0.0,1.0,0.0,0.0
